In [1]:
from collections import defaultdict
from scipy import stats
from scipy.stats import erlang, expon, norm 
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json

In [11]:
class abstractRV(stats.rv_continuous):
    def __init__(self, xtol=1e-14, seed=None, cdf=lambda x: 2):
        super().__init__(a=0, xtol=xtol, seed=seed)
        self._custom_cdf = cdf

    def _cdf(self, x):
        self._custom_cdf(x)

In [26]:
p = abstractRV(cdf= lambda x: 1-np.exp(-x**2))

In [27]:
p.rvs(size=1000)

ValueError: The function value at x=10.0 is NaN; solver cannot continue.

# Implement Notes

We can actually implement multiple src nodes

This works for arbitary graphs with arbitrary node names

# Destination
WHen ddid each observer get infected?

Try to emulate Devlin's data-structure of trees but in general for graphs

In [7]:
class Graph(object):

    def __init__(self, edge_json, node_size=None, directed=False):
        # graph data structures
        self._graph = defaultdict(set)
        # Node information
        self._infected = defaultdict(lambda : False)
        self._parent = defaultdict(lambda : None)
        self._node_infect_time = defaultdict(lambda : 0) # to exploit "memoryless" exponential RV (s + t - s)
        self.edge_set = self.make_edge_set(edge_json)
        # Graph creation
        self._directed = directed
        self.add_connections(self.edge_set)
        self._adjency_matrix = self.construct_matrix(self.edge_set)
        
    def add_connections(self, edge_set):
        for node1, node2, wt in edge_set:
            self.add_edge(node1,node2, wt)
            
    def add_edge(self, src, dst, wt):
        self._graph[src].add((dst, wt))
        if self._directed == False:
            self._graph[dst].add((src, wt))
            
    def construct_matrix(self, edge_set):
        df = pd.DataFrame(edge_set)
        df = df.pivot(index=0, columns=1, values=2)
        if self._directed == False:
            df = df.combine_first(df.T)
        if self._directed == True:
            idx = df.columns.union(df.index)
            df = df.reindex(index = idx, columns=idx, fill_value=np.inf)
        df = df.fillna(np.inf)
        return df

    # Time evolution simulation, based on dict graph
    def simulate_gossip(self, src, dst, time_step = 0.1):
        self.reset_simulation() 

        src = np.array([src]).flatten() # transform scalars and lists to iterables
        dst = np.array([src]).flatten()
        global_t = 0

        for node in src:
            if self._graph[node] == set(): 
                raise ValueError(f"Source node {src} is not in the graph")
            self._infected[src] = True
            self._node_infect_time[src] = 0
        for node in dst:
            if self._graph[node] == set():
                raise ValueError(f"Destination node {dst} is not in the graph")

        # DOesn't check past source for graph traversibility
        while not self._infected[dst]:
            global_t = global_t + time_step
            current_tick_infected = [key for key in self._infected.keys() if self._infected[key] == True]
            for infected in current_tick_infected:
                for new_infection, wt in self._graph[infected]:
                    if not self._infected[new_infection]:
                        infection_event_probability = wt(global_t - self._node_infect_time[infected])
                        self._infected[new_infection] = np.random.choice([True, False], p=[infection_event_probability, 1-infection_event_probability])
                        if self._infected[new_infection]:
                            self._parent[new_infection] = infected
                            self._node_infect_time[new_infection] = global_t
                            
        return global_t
    
    def simulate_gossip_df(self, src, dst, time_step=0.1):
        self.reset_simulation()

        src = np.array([src]).flatten() # transform scalars and lists to iterables
        dst = np.array([dst]).flatten()
        global_t = 0
        
        for node in src:
            self._infected[node] = True
            self._node_infect_time[node] = 0
            
        for node in dst:
            if self._graph[node] == set():
                raise ValueError(f"Destination node {dst} is not in the graph")

        if ((g._adjency_matrix.loc[src] == np.inf).all()).all():
            raise ValueError(f"Source node is not in the graph")
            
        terminate = False
        
        while not terminate:
            global_t = global_t + time_step
            current_tick_infected = [key for key in self._infected.keys() if self._infected[key] == True]
            if ((g._adjency_matrix.loc[np.array(current_tick_infected)] == np.inf).all()).all():
                raise ValueError("No path to dst")
            for infected in current_tick_infected:
                for col, item in enumerate(self._adjency_matrix.loc[infected]):
                    if item == np.inf:
                        continue
                    new_infection = self._adjency_matrix.columns[col]
                    if not self._infected[new_infection]:
                        infection_event_probability = item(global_t - self._node_infect_time[infected])
                        self._infected[new_infection] = np.random.choice([True, False], p=[infection_event_probability, 1-infection_event_probability])
                        if self._infected[new_infection]:
                            self._parent[new_infection] = infected
                            self._node_infect_time[new_infection] = global_t
                            self._adjency_matrix.loc[infected, new_infection] = np.inf
                            if (self._directed == False):
                                self._adjency_matrix.loc[new_infection, infected] = np.inf
                            if (new_infection in dst):
                                return global_t

    
    # Make sure graph is using scipy.stats library
    def simulate_gossip_rv(self, src, dst):
        self.reset_simulation()

        src = np.array([src]).flatten() # transform scalars and lists to iterables
        dst = np.array([src]).flatten()
        global_t = 0
        
        for node in src:
            self._infected[node] = True
            self._node_infect_time[node] = 0
            
        if ((g._adjency_matrix.loc[src] == np.inf).all()).all():
            raise ValueError(f"Source node {src} is not in the graph")
        
        terminate = False
        while not terminate:
            current_tick_infected = [key for key in self._infected.keys() if self._infected[key] == True]
            if ((g._adjency_matrix.loc[np.array(current_tick_infected)] == np.inf).all()).all():
                raise ValueError("No path to dst")
            global_t = 0
            for infected in current_tick_infected:
                min_node = None
                min_infect_time = np.inf
                for col, rv_new_infection in enumerate(self._adjency_matrix.loc[infected]):
                    new_infection = self._adjency_matrix.columns[col]
            
            
        return None

    def reset_simulation(self):
        for key in self._graph.keys():
            self._infected[key] = False
            self._parent[key] = None
            self._node_infect_time[key] = 0
        self._adjency_matrix = self.construct_matrix(self.edge_set)

    def construct_path(self, dst):
        path = []
        curr_node = dst
        while curr_node is not None:
            path.append(curr_node)
            curr_node = self._parent[curr_node]
        
        return path
    
    def make_edge_set(self, edge_json):
        edge_set = set()
        for key, value in edge_json.items():
            edges = key.split(',')
            distribution = self.process_distribution_params(value)
            edge_tuple = (edges[0], edges[1], distribution)
            edge_set.add(edge_tuple)
        return edge_set
                               
    def process_distribution_params(self, function_dict):
        distribution_map = {
            "E" : expon, # assuming params lambda = 1.0
            "N" : norm, # assuming unit normal
            "custom" : None, # customRV, # not working
        }
        distribution = distribution_map[function_dict["distribution"]]
        return distribution
                               


In [13]:
exp_t = lambda t: 1 - np.e**(-t)

edge_set = {
    "1,2": { "distribution": "E", "parameters": { "lambda" : 1.0 }}, 
    "2,1": { "distribution": "E", "parameters": { "lambda" : 1.0 }}, 
    "1,3": { "distribution": "E", "parameters": { "lambda" : 1.0 }}, 
    "2,4": { "distribution": "E", "parameters": { "lambda" : 1.0 }}, 
    "2,5": { "distribution": "E", "parameters": { "lambda" : 1.0 }}, 
    "3,6": { "distribution": "E", "parameters": { "lambda" : 1.0 }}, 
    "4,5": { "distribution": "E", "parameters": { "lambda" : 1.0 }}, 
    "4,6": { "distribution": "E", "parameters": { "lambda" : 1.0 }}, 
}

g = Graph(edge_set)#, directed=True)
g._adjency_matrix

,1,2,3,4,5,6
1,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf,inf
2,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...,inf
3,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...
4,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...
5,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf
6,inf,inf,<scipy.stats._continuous_distns.expon_gen obje...,<scipy.stats._continuous_distns.expon_gen obje...,inf,inf


In [ ]:
edge_set_obs = {
    "1,2": { "distribution": "custom", "parameters": { "cdf" : exp_t }}, 
    "1,3": { "distribution": "custom", "parameters": { "cdf" : exp_t }}, 
    "2,4": { "distribution": "custom", "parameters": { "cdf" : exp_t }}, 
    "2,5": { "distribution": "custom", "parameters": { "cdf" : exp_t }}, 
    "3,6": { "distribution": "custom", "parameters": { "cdf" : exp_t }}, 
    "4,5": { "distribution": "custom", "parameters": { "cdf" : exp_t }}, 
    "4,6": { "distribution": "custom", "parameters": { "cdf" : exp_t }}, 
}

In [ ]:
((g._adjency_matrix.loc[np.array([1,2])] == np.inf).all()).all()
# (g._adjency_matrix.loc[np.array([1,2])] == np.inf).all()
# g._adjency_matrix.loc[np.array([1,2])] == np.inf

np.False_

In [ ]:
g._adjency_matrix

,1,2,3,4,5,6
1,inf,<function <lambda> at 0x7f62d7590670>,<function <lambda> at 0x7f62d7590670>,inf,inf,inf
2,<function <lambda> at 0x7f62d7590670>,inf,inf,<function <lambda> at 0x7f62d7590670>,<function <lambda> at 0x7f62d7590670>,inf
3,<function <lambda> at 0x7f62d7590670>,inf,inf,inf,inf,<function <lambda> at 0x7f62d7590670>
4,inf,<function <lambda> at 0x7f62d7590670>,inf,inf,<function <lambda> at 0x7f62d7590670>,<function <lambda> at 0x7f62d7590670>
5,inf,<function <lambda> at 0x7f62d7590670>,inf,<function <lambda> at 0x7f62d7590670>,inf,inf
6,inf,inf,<function <lambda> at 0x7f62d7590670>,<function <lambda> at 0x7f62d7590670>,inf,inf


In [ ]:
t = g.simulate_gossip_df(1,6,time_step=10**(-4))
path = g.construct_path(6)
display(path)
print(t)
g._adjency_matrix

[6, 3, 1]

0.024999999999999904


,1,2,3,4,5,6
0,,,,,,
1,inf,inf,inf,inf,inf,inf
2,inf,inf,inf,<function <lambda> at 0x7f62d7590670>,<function <lambda> at 0x7f62d7590670>,inf
3,inf,inf,inf,inf,inf,inf
4,inf,<function <lambda> at 0x7f62d7590670>,inf,inf,<function <lambda> at 0x7f62d7590670>,<function <lambda> at 0x7f62d7590670>
5,inf,<function <lambda> at 0x7f62d7590670>,inf,<function <lambda> at 0x7f62d7590670>,inf,inf
6,inf,inf,inf,<function <lambda> at 0x7f62d7590670>,inf,inf


In [ ]:
g._adjency_matrix

,1,2,3,4,5,6
0,,,,,,
1,inf,inf,inf,inf,inf,inf
2,inf,inf,inf,<function <lambda> at 0x7f62d7590670>,<function <lambda> at 0x7f62d7590670>,inf
3,inf,inf,inf,inf,inf,inf
4,inf,<function <lambda> at 0x7f62d7590670>,inf,inf,<function <lambda> at 0x7f62d7590670>,<function <lambda> at 0x7f62d7590670>
5,inf,<function <lambda> at 0x7f62d7590670>,inf,<function <lambda> at 0x7f62d7590670>,inf,inf
6,inf,inf,inf,<function <lambda> at 0x7f62d7590670>,inf,inf
